# Immunogenicity tool test: Testing setttings for tools: NetMHCpan_Immunogen: Tool outputs
Author: Rebecka Antonsson\
Version: 1 (26-02-2026)

# Notebook explanation
Notebook to calculate the scores of the outputs from the tool BioPhi(OASis) with different settings.

Settings tested are:
    immunogen_27allels_peplength8-14


Immunogenicity score above 0 means that peptide/allele combination is more likely than not to elicit a immune response. A higher score is therefore assumed to mean more immune response.



For each setting three measurments on how good they perform will be calculated:
    1. Mean absolute rank error (MARE) for antibodies
    2. Spearman rank correlation
    3. Mean absolute rank error for the two nanobodies

1. MARE is just the absolute difference between the known rank of the antibody and the predicted rank
2. Spearman rank correlation is a statistical test that can compare two lists of ranking and tell how well they align. 1 is a perfect correlation
0 means no relation between the two variables (no correlation, random) and -1 is a perfect reversed correlation (very bad).
3. Same calculation as for 1, but separated from the rest. Here a separate ranking where the 2 nanobodies are included are performed, hence antobodies and nanobodies get a number 1 to 39 and the MARE for the nanobodies is calculated. 

In [91]:
import pandas as pd
from scipy.stats import spearmanr

In [92]:
immunogen_peplength8_14 = pd.read_csv("tool_outputs/immunogen_peplength8-14_26-02-2026.csv")
seqs_immunogen_peplength8_14 = pd.read_csv("tool_outputs/sequence_table_immunogen_peplength8-14_26-02-2026.csv")

In [93]:
immunogen_peplength8_14.head()
seqs_immunogen_peplength8_14.head()

,seq #,sequence name,sequence
0,1,BEZLOTOXUMAB,EVQLVQSGAEVKKSGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...
1,2,VISILIZUMAB,QVQLVQSGAEVKKPGASVKVSCKASGYTFISYTMHWVRQAPGQGLE...
2,3,OMALIZUMAB,EVQLVESGGGLVQPGGSLRLSCAVSGYSITSGYSWNWIRQAPGKGL...
3,4,EVOLOCUMAB,EVQLVQSGAEVKKPGASVKVSCKASGYTLTSYGISWVRQAPGQGLE...
4,5,SECUKINUMAB,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYWMNWVRQAPGKGLE...


In [94]:
# print number of unique values in seq # column (should be 39)
print(immunogen_peplength8_14['seq #'].nunique())
# print number of unique valies in HLA allele column (should be 27)
print(immunogen_peplength8_14['allele'].nunique())
# Print number of rows
print(immunogen_peplength8_14.shape[0])

39
27
1594971


In [95]:
# for loop, from 1 to length of unique values in seq # column, 
# #print number of rows that have wat is defined as a immunigenic score 
unique_seq_numbers = immunogen_peplength8_14['seq #'].unique()
for seq_number in sorted(unique_seq_numbers):
    subset = immunogen_peplength8_14[immunogen_peplength8_14['seq #'] == seq_number]
    count_immunogen = subset[subset['immunogenicity score'] > 0].shape[0]
    total_count = subset.shape[0]
    # take the count_immunogen and divide by the total number of rows in the subset, and print the result. ANd multiply with 100 to get the percantage
    print((count_immunogen/total_count)*100)

37.59052007899934
32.93650793650794
38.35263835263835
28.371161548731642
50.031746031746025
42.077922077922075
39.349489795918366
37.745740498034074
51.598173515981735
29.24847664184157
35.78042328042328
43.78737541528239
48.401826484018265
42.5888665325285
36.74388674388675
34.53473132372215
38.095238095238095
44.2792501616031
39.285714285714285
44.07431133888533
38.72870249017038
46.54594232059021
33.2010582010582
41.409691629955944
41.05590062111801
41.4486921529175
43.90084801043705
40.36697247706422
37.692823608316566
43.84010484927916
43.97116644823067
43.05106658047835
33.77221856484529
40.42132982225149
50.531286894923255
39.136212624584715
41.9397116644823
56.295399515738495
39.12768647281922


In [96]:
# get unique seq # values and unique peptide length values
unique_seq_numbers = immunogen_peplength8_14['seq #'].unique()
unique_peptide_lengths = immunogen_peplength8_14['peptide length'].unique()

# Make a dataframe with one colums as seq, with all the unique seq # values. 
# In the for loop one wolumn for each peptide length and the percent immugen will be added 
calc_rank_table = pd.DataFrame({'seq#': sorted(unique_seq_numbers)})

# for loop, from 1 to length of unique values in seq # column, 
# #print number of rows that have wat is defined as a immunigenic score 

for peptide_length in sorted(unique_peptide_lengths):
    for seq_number in sorted(unique_seq_numbers):
        # subset unique seq # and peptide length combination
        subset = immunogen_peplength8_14[(immunogen_peplength8_14['seq #'] == seq_number) & (immunogen_peplength8_14['peptide length'] == peptide_length)]
        count_immunogen = subset[subset['immunogenicity score'] > 0].shape[0]
        total_count = subset.shape[0]
        # take the count_immunogen and divide by the total number of rows in the subset, and print the result. ANd multiply with 100 to get the percantage
        percent_immunogen = (count_immunogen/total_count)*100
        # add the percent immunogen to the dataframe, with the column name as the peptide length
        calc_rank_table.loc[calc_rank_table['seq#'] == seq_number, f'peptide length {peptide_length}'] = percent_immunogen

# add a cloumn with the average percent immunogen for each seq #, and name it all_pep_lengths
calc_rank_table['all_pep_lengths'] = calc_rank_table.filter(regex='peptide length').mean(axis=1)

calc_rank_table


,seq#,peptide length 8,peptide length 9,peptide length 10,peptide length 11,peptide length 12,peptide length 13,peptide length 14,all_pep_lengths
0,1,42.272727,38.812785,37.155963,36.405530,37.037037,35.348837,35.981308,37.573456
1,2,35.616438,33.944954,34.101382,32.407407,31.162791,31.775701,31.455399,32.923439
2,3,41.777778,39.732143,38.116592,38.738739,38.914027,35.909091,35.159817,38.335455
3,4,30.875576,29.629630,27.906977,29.439252,29.107981,25.943396,25.592417,28.356461
4,5,52.192982,51.101322,50.442478,49.333333,49.553571,48.878924,48.648649,50.021608
5,6,46.636771,45.945946,44.343891,41.818182,40.639269,37.614679,37.327189,42.046561
6,7,40.969163,42.035398,41.333333,41.517857,39.013453,36.036036,34.389140,39.327769
7,8,42.081448,40.000000,40.639269,37.614679,36.866359,35.185185,31.627907,37.716407
8,9,50.000000,49.321267,51.363636,52.968037,51.834862,53.917051,51.851852,51.608101
9,10,31.775701,31.924883,30.188679,30.805687,29.047619,27.751196,23.076923,29.224384


In [97]:
# both the dataframes calc_rank_table and the seqs_immunogen_peplength8_14 have a seq # column,
# Instert the "sequence name" to the calc_rank_table, at the matching seq #
calc_rank_table = calc_rank_table.merge(seqs_immunogen_peplength8_14[['seq #', 'sequence name']], left_on='seq#', right_on='seq #', how='left')

calc_rank_table

,seq#,peptide length 8,peptide length 9,peptide length 10,peptide length 11,peptide length 12,peptide length 13,peptide length 14,all_pep_lengths,seq #,sequence name
0,1,42.272727,38.812785,37.155963,36.405530,37.037037,35.348837,35.981308,37.573456,1,BEZLOTOXUMAB
1,2,35.616438,33.944954,34.101382,32.407407,31.162791,31.775701,31.455399,32.923439,2,VISILIZUMAB
2,3,41.777778,39.732143,38.116592,38.738739,38.914027,35.909091,35.159817,38.335455,3,OMALIZUMAB
3,4,30.875576,29.629630,27.906977,29.439252,29.107981,25.943396,25.592417,28.356461,4,EVOLOCUMAB
4,5,52.192982,51.101322,50.442478,49.333333,49.553571,48.878924,48.648649,50.021608,5,SECUKINUMAB
5,6,46.636771,45.945946,44.343891,41.818182,40.639269,37.614679,37.327189,42.046561,6,DENOSUMAB
6,7,40.969163,42.035398,41.333333,41.517857,39.013453,36.036036,34.389140,39.327769,7,IBALIZUMAB
7,8,42.081448,40.000000,40.639269,37.614679,36.866359,35.185185,31.627907,37.716407,8,OCRELIZUMAB
8,9,50.000000,49.321267,51.363636,52.968037,51.834862,53.917051,51.851852,51.608101,9,FREMANEZUMAB
9,10,31.775701,31.924883,30.188679,30.805687,29.047619,27.751196,23.076923,29.224384,10,BASILIXIMAB


In [98]:
# create a copy of the calc_rank_table, without the two rows with sequence name "Caplacizumab" and "Vobarilizumab"
calc_rank_table_AB = calc_rank_table[~calc_rank_table['sequence name'].isin(['Caplacizumab', 'Vobarilizumab'])]

calc_rank_table_AB

,seq#,peptide length 8,peptide length 9,peptide length 10,peptide length 11,peptide length 12,peptide length 13,peptide length 14,all_pep_lengths,seq #,sequence name
0,1,42.272727,38.812785,37.155963,36.405530,37.037037,35.348837,35.981308,37.573456,1,BEZLOTOXUMAB
1,2,35.616438,33.944954,34.101382,32.407407,31.162791,31.775701,31.455399,32.923439,2,VISILIZUMAB
2,3,41.777778,39.732143,38.116592,38.738739,38.914027,35.909091,35.159817,38.335455,3,OMALIZUMAB
3,4,30.875576,29.629630,27.906977,29.439252,29.107981,25.943396,25.592417,28.356461,4,EVOLOCUMAB
4,5,52.192982,51.101322,50.442478,49.333333,49.553571,48.878924,48.648649,50.021608,5,SECUKINUMAB
5,6,46.636771,45.945946,44.343891,41.818182,40.639269,37.614679,37.327189,42.046561,6,DENOSUMAB
6,7,40.969163,42.035398,41.333333,41.517857,39.013453,36.036036,34.389140,39.327769,7,IBALIZUMAB
7,8,42.081448,40.000000,40.639269,37.614679,36.866359,35.185185,31.627907,37.716407,8,OCRELIZUMAB
8,9,50.000000,49.321267,51.363636,52.968037,51.834862,53.917051,51.851852,51.608101,9,FREMANEZUMAB
9,10,31.775701,31.924883,30.188679,30.805687,29.047619,27.751196,23.076923,29.224384,10,BASILIXIMAB


In [99]:
# for each column from peptide length 8 to all_pep_lengths, make a new dataframe witht the sequence name, peptide column
# name the dataframe as the column with the peptide length, keep the name of the sequence column, but rename the petide colum to "percent immunogenicity"

peptide_length_8 = calc_rank_table_AB[['sequence name', 'peptide length 8']].rename(columns={'peptide length 8': 'percent immunogenicity'})
peptide_length_9 = calc_rank_table_AB[['sequence name', 'peptide length 9']].rename(columns={'peptide length 9': 'percent immunogenicity'})
peptide_length_10 = calc_rank_table_AB[['sequence name', 'peptide length 10']].rename(columns={'peptide length 10': 'percent immunogenicity'})
peptide_length_11 = calc_rank_table_AB[['sequence name', 'peptide length 11']].rename(columns={'peptide length 11': 'percent immunogenicity'})
peptide_length_12 = calc_rank_table_AB[['sequence name', 'peptide length 12']].rename(columns={'peptide length 12': 'percent immunogenicity'})
peptide_length_13 = calc_rank_table_AB[['sequence name', 'peptide length 13']].rename(columns={'peptide length 13': 'percent immunogenicity'})
peptide_length_14 = calc_rank_table_AB[['sequence name', 'peptide length 14']].rename(columns={'peptide length 14': 'percent immunogenicity'})
all_pep_lengths = calc_rank_table_AB[['sequence name', 'all_pep_lengths']].rename(columns={'all_pep_lengths': 'percent immunogenicity'})



In [100]:
ADA_rank = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'Caplacizumab':28,
'LANADELUMAB':29,
'BUROSUMAB':30,
'BENRALIZUMAB':31,
'ADALIMUMAB':32,
'IXEKIZUMAB':33,
'RITUXIMAB':34,
'INFLIXIMAB':35,
'GOLIMUMAB':36,
'Vobarilizumab':37,
'BOCOCIZUMAB':38,
'ALEMTUZUMAB':39
}

# create antoher dictionary where the two nanobdoies are removed 
ADA_rank_AB = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'LANADELUMAB':28,
'BUROSUMAB':29,
'BENRALIZUMAB':30,
'ADALIMUMAB':31,
'IXEKIZUMAB':32,
'RITUXIMAB':33,
'INFLIXIMAB':34,
'GOLIMUMAB':35,
'BOCOCIZUMAB':36,
'ALEMTUZUMAB':37
}

In [101]:
all_dfs_AB = {
'peptide length 8': peptide_length_8,
'peptide length 9': peptide_length_9,
'peptide length 10': peptide_length_10, 
'peptide length 11': peptide_length_11,
'peptide length 12': peptide_length_12,
'peptide length 13': peptide_length_13,
'peptide length 14': peptide_length_14,
'all peptide lengths': all_pep_lengths
}

In [102]:
# Add the ADA_rank_AB to the AB dataframes, map them by the antibody name
for df in all_dfs_AB.values():
    df['ADA_rank'] = df['sequence name'].map(ADA_rank_AB)

# Rank the percent immunogenicity scores from 1 to 37
for df in all_dfs_AB.values():
    df['ranked_by_tool'] = df['percent immunogenicity'].rank(ascending=True, method='average')

# Add a column called MARE, for each row add the value for the absolute difference between the ranked_by_tool and ADA_rank
for df in all_dfs_AB.values():
    df['MARE'] = (df['ranked_by_tool'] - df['ADA_rank']).abs()

In [ ]:
# Do all those steps for the dataframes with nanobodies as well

# remove the two columns called seq # from the calc_rank_table
calc_rank_table = calc_rank_table.drop(columns=['seq #', 'seq#'])

# rename all columns to "Imunogen_score" + the peptide length only the number at the end, for the last column called "all_pep_lengths" rename it to "Immunogen_score_all"
new_column_names = {}
for col in calc_rank_table.columns:
    if col.startswith('peptide length'):
        pep_length = col.split(' ')[-1]
        new_column_names[col] = f'Immunogen_score_PL{pep_length}'
    elif col == 'all_pep_lengths':
        new_column_names[col] = 'Immunogen_score_all'
calc_rank_table = calc_rank_table.rename(columns=new_column_names)  

# Rename the column "sequence name" to "Antibody", 
calc_rank_table.rename(columns={'sequence name': 'Antibody'}, inplace=True)
# Place the anotbody columns as the first column beacuse its easier to read
calc_rank_table = calc_rank_table.iloc[:, [8] + list(range(0, 8))]

,Antibody,Immunogen_score_PL8,Immunogen_score_PL9,Immunogen_score_PL10,Immunogen_score_PL11,Immunogen_score_PL12,Immunogen_score_PL13,Immunogen_score_PL14,Immunogen_score_all
0,BEZLOTOXUMAB,42.272727,38.812785,37.155963,36.405530,37.037037,35.348837,35.981308,37.573456
1,VISILIZUMAB,35.616438,33.944954,34.101382,32.407407,31.162791,31.775701,31.455399,32.923439
2,OMALIZUMAB,41.777778,39.732143,38.116592,38.738739,38.914027,35.909091,35.159817,38.335455
3,EVOLOCUMAB,30.875576,29.629630,27.906977,29.439252,29.107981,25.943396,25.592417,28.356461
4,SECUKINUMAB,52.192982,51.101322,50.442478,49.333333,49.553571,48.878924,48.648649,50.021608
5,DENOSUMAB,46.636771,45.945946,44.343891,41.818182,40.639269,37.614679,37.327189,42.046561
6,IBALIZUMAB,40.969163,42.035398,41.333333,41.517857,39.013453,36.036036,34.389140,39.327769
7,OCRELIZUMAB,42.081448,40.000000,40.639269,37.614679,36.866359,35.185185,31.627907,37.716407
8,FREMANEZUMAB,50.000000,49.321267,51.363636,52.968037,51.834862,53.917051,51.851852,51.608101
9,BASILIXIMAB,31.775701,31.924883,30.188679,30.805687,29.047619,27.751196,23.076923,29.224384


In [ ]:
# for each column staring with Immunogen_score, add a column called tool_rank, with what comes after the last _
# fill this column with the rank of the immunogenicity score, from 1 to 39, with 39 being the most immunogenic 
for col in calc_rank_table.columns:
    if col.startswith('Immunogen_score'):
        tool_rank_col_name = f'tool_rank_{col.split("_")[-1]}'
        calc_rank_table[tool_rank_col_name] = calc_rank_table[col].rank(ascending=True, method='average')

# Add the ADA_rank to the calc_rank_table, map them by the antibody name
calc_rank_table['ADA_rank'] = calc_rank_table['Antibody'].map(ADA_rank)

# Calculate the MARE for only the two rows with Antibody name "Caplacizumab" and "Vobarilizumab
# ", for each columns starting with tool_rank and the ADA_rank, add a column called MARE_+ the peptide length,

for col in calc_rank_table.columns:
    if col.startswith('tool_rank_'):
        pep_length = col.split('_')[-1]
        calc_rank_table[f'MARE_{pep_length}'] = (calc_rank_table[col] - calc_rank_table['ADA_rank']).abs()

calc_rank_table

,Antibody,Immunogen_score_PL8,Immunogen_score_PL9,Immunogen_score_PL10,Immunogen_score_PL11,Immunogen_score_PL12,Immunogen_score_PL13,Immunogen_score_PL14,Immunogen_score_all,ADA_rank,...,tool_rank_PL14,tool_rank_all,MARE_PL8,MARE_PL9,MARE_PL10,MARE_PL11,MARE_PL12,MARE_PL13,MARE_PL14,MARE_all
0,BEZLOTOXUMAB,42.272727,38.812785,37.155963,36.405530,37.037037,35.348837,35.981308,37.573456,1,...,16.0,9.0,20.0,8.0,8.0,7.0,11.0,9.0,15.0,8.0
1,VISILIZUMAB,35.616438,33.944954,34.101382,32.407407,31.162791,31.775701,31.455399,32.923439,2,...,5.0,3.0,1.5,1.0,4.0,2.0,1.0,2.0,3.0,1.0
2,OMALIZUMAB,41.777778,39.732143,38.116592,38.738739,38.914027,35.909091,35.159817,38.335455,3,...,14.0,13.0,16.0,10.0,7.5,10.0,13.0,8.0,11.0,10.0
3,EVOLOCUMAB,30.875576,29.629630,27.906977,29.439252,29.107981,25.943396,25.592417,28.356461,4,...,2.0,1.0,3.0,3.0,3.0,3.0,2.0,3.0,2.0,3.0
4,SECUKINUMAB,52.192982,51.101322,50.442478,49.333333,49.553571,48.878924,48.648649,50.021608,5,...,36.0,36.0,34.0,33.0,30.0,31.0,31.0,30.0,31.0,31.0
5,DENOSUMAB,46.636771,45.945946,44.343891,41.818182,40.639269,37.614679,37.327189,42.046561,6,...,19.0,25.0,27.0,27.0,22.0,17.0,16.0,10.0,13.0,19.0
6,IBALIZUMAB,40.969163,42.035398,41.333333,41.517857,39.013453,36.036036,34.389140,39.327769,7,...,11.0,18.0,7.0,15.0,11.0,14.0,10.0,6.0,4.0,11.0
7,OCRELIZUMAB,42.081448,40.000000,40.639269,37.614679,36.866359,35.185185,31.627907,37.716407,8,...,6.0,11.0,12.0,6.0,8.0,2.0,2.5,1.0,2.0,3.0
8,FREMANEZUMAB,50.000000,49.321267,51.363636,52.968037,51.834862,53.917051,51.851852,51.608101,9,...,38.0,38.0,28.0,27.0,28.0,29.0,29.0,29.0,29.0,29.0
9,BASILIXIMAB,31.775701,31.924883,30.188679,30.805687,29.047619,27.751196,23.076923,29.224384,10,...,1.0,2.0,8.0,8.0,8.0,8.0,9.0,8.0,9.0,8.0


In [ ]:
#

In [104]:
# Create a new dataframe with the name of the dataframe as a row, the sum MARE, nanobody mean MARE and the spearman rank correlation
ranked_score_table = pd.DataFrame({
    # give it the same keys as in the all_dfs dictionary
    "Dataframe": list(all_dfs_AB.keys()),
    "AB_sum_MARE": [df['MARE'].sum() for df in all_dfs_AB.values()] ,
    nanobody_mean_MARE': [df[df['Antibody'].isin(['Caplacizumab', 'Vobarilizumab'])]['MARE'].mean() for df in all_dfs.values()],
    "spearmanr": [spearmanr(df['ranked_by_tool'], df['ADA_rank']).correlation for df in all_dfs_AB.values()],
    })

ranked_score_table

SyntaxError: unterminated string literal (detected at line 6) (200893935.py, line 6)